# Sephora Product Intelligence — SQL Layer

## Purpose
Loading all processed datasets into a SQLite database and running 
analytics queries. This creates a proper data model that can connect 
to Tableau or Power BI for dashboard building.

## Tables loaded:
- products_clean — all 1,952 skincare products
- reviews_clean — 980,344 individual reviews
- features_ml — 1,915 products with engineered features
- results_reliable — ML predictions with overrated/underrated flags
- feature_importance — what drives product ratings
- top_products_by_category — best products per subcategory

## 1. Setup & Imports

In [12]:
import pandas as pd
import numpy as np
import sqlite3
import os
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)

print("Libraries loaded")

Libraries loaded


## 2. Load Processed Data

In [13]:
# Load all processed datasets
products_clean = pd.read_csv("../data/processed/products_clean.csv")
reviews_clean = pd.read_csv("../data/processed/reviews_clean.csv")
features_ml = pd.read_csv("../data/processed/features_ml.csv")
results_reliable = pd.read_csv("../data/processed/results_reliable.csv")
feature_importance = pd.read_csv("../data/processed/feature_importance.csv")
top_products = pd.read_csv("../data/processed/top_products_by_category.csv")

print("=== DATASETS LOADED ===")
print(f"products_clean: {products_clean.shape}")
print(f"reviews_clean: {reviews_clean.shape}")
print(f"features_ml: {features_ml.shape}")
print(f"results_reliable: {results_reliable.shape}")
print(f"feature_importance: {feature_importance.shape}")
print(f"top_products: {top_products.shape}")

=== DATASETS LOADED ===
products_clean: (1952, 22)
reviews_clean: (980344, 19)
features_ml: (1915, 28)
results_reliable: (1456, 14)
feature_importance: (21, 2)
top_products: (80, 14)


## 3. Create SQLite Database
Loading all tables into a single SQLite database file.
This is our Load step — completing the full ETL pipeline.

In [14]:
# Create database in project root
db_path = "../sephora_bi.db"

# Connect to SQLite — creates file if it doesn't exist
conn = sqlite3.connect(db_path)

# Load all tables
products_clean.to_sql("products_clean", conn, if_exists="replace", index=False)
reviews_clean.to_sql("reviews_clean", conn, if_exists="replace", index=False)
features_ml.to_sql("features_ml", conn, if_exists="replace", index=False)
results_reliable.to_sql("results_reliable", conn, if_exists="replace", index=False)
feature_importance.to_sql("feature_importance", conn, if_exists="replace", index=False)
top_products.to_sql("top_products_by_category", conn, if_exists="replace", index=False)

print("=== DATABASE CREATED ===")
print(f"Location: {os.path.abspath(db_path)}")

# Verify tables
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print(f"\nTables in database:")
for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table[0]}")
    count = cursor.fetchone()[0]
    print(f"  {table[0]}: {count:,} rows")

=== DATABASE CREATED ===
Location: /Users/athenadjojonegoro/Desktop/sephora-bi-project/sephora_bi.db

Tables in database:
  products_clean: 1,952 rows
  reviews_clean: 980,344 rows
  features_ml: 1,915 rows
  results_reliable: 1,456 rows
  feature_importance: 21 rows
  top_products_by_category: 80 rows


## 4. SQL Analytics Queries

Running analytics directly on the SQLite database using pandas read_sql.
These queries answer our core business questions and will power the dashboard.

### Query 1 — Brand Performance
Which brands consistently deliver the best rated products?

In [15]:
query1 = """
SELECT 
    brand_name,
    COUNT(product_id) AS total_products,
    ROUND(AVG(avg_rating), 3) AS avg_rating,
    ROUND(AVG(price_usd), 2) AS avg_price,
    ROUND(AVG(ingredient_score), 3) AS avg_ingredient_score,
    ROUND(AVG(five_star_share), 3) AS avg_five_star_share,
    ROUND(AVG(review_count), 0) AS avg_review_count
FROM features_ml
GROUP BY brand_name
HAVING COUNT(product_id) >= 3
ORDER BY avg_rating DESC
LIMIT 20
"""

brand_performance = pd.read_sql(query1, conn)
print("=== TOP 20 BRANDS BY RATING (3+ products) ===")
print(brand_performance.to_string(index=False))

=== TOP 20 BRANDS BY RATING (3+ products) ===
       brand_name  total_products  avg_rating  avg_price  avg_ingredient_score  avg_five_star_share  avg_review_count
      The Nue Co.               3       5.000     57.330                 0.205                1.000            13.000
             Mara              10       4.816     65.800                 0.085                0.876            94.000
  Macrene Actives               5       4.788    147.000                 0.128                0.844            65.000
           Damdam               7       4.718     49.000                 0.109                0.791           161.000
          Facegym               8       4.695     72.380                 0.121                0.786            68.000
 Rose Ingleton Md               7       4.694     64.430                 0.205                0.801            35.000
        Hourglass               9       4.679     86.670                 0.066                0.819            46.000
Benefit Co

### Query 2 — Category Summary
How do skincare subcategories compare on rating, price, and ingredients?

In [16]:
query2 = """
SELECT
    secondary_category,
    COUNT(product_id) AS total_products,
    ROUND(AVG(avg_rating), 3) AS avg_rating,
    ROUND(AVG(price_usd), 2) AS avg_price,
    ROUND(AVG(ingredient_score), 3) AS avg_ingredient_score,
    ROUND(AVG(review_count), 0) AS avg_review_count,
    ROUND(AVG(five_star_share), 3) AS avg_five_star_share,
    ROUND(AVG(recommendation_rate), 3) AS avg_recommendation_rate
FROM features_ml
GROUP BY secondary_category
ORDER BY avg_rating DESC
"""

category_summary = pd.read_sql(query2, conn)
print("=== CATEGORY PERFORMANCE SUMMARY ===")
print(category_summary.to_string(index=False))

=== CATEGORY PERFORMANCE SUMMARY ===
    secondary_category  total_products  avg_rating  avg_price  avg_ingredient_score  avg_review_count  avg_five_star_share  avg_recommendation_rate
             Cleansers             354       4.314     35.310                 0.084           567.000                0.660                    0.856
            Treatments             452       4.304     68.180                 0.138           491.000                0.647                    0.856
          Moisturizers             538       4.275     72.650                 0.089           553.000                0.641                    0.845
                 Masks             165       4.250     46.490                 0.085           427.000                0.639                    0.842
Lip Balms & Treatments              61       4.147     24.810                 0.048          1011.000                0.622                    0.792
             Sunscreen             108       4.122     43.560              

### Query 3 — Overrated vs Underrated Products
Full list of flagged products with all key metrics.

In [17]:
query3 = """
SELECT
    product_name,
    brand_name,
    secondary_category,
    price_usd,
    price_tier,
    ROUND(avg_rating, 3) AS actual_rating,
    ROUND(predicted_rating, 3) AS predicted_rating,
    ROUND(gap, 3) AS gap,
    flag,
    review_count,
    ROUND(ingredient_score, 3) AS ingredient_score
FROM results_reliable
WHERE flag != 'Normal'
ORDER BY gap ASC
"""

flagged_products = pd.read_sql(query3, conn)
print(f"Total flagged products: {len(flagged_products)}")
print(f"\nOverrated: {len(flagged_products[flagged_products['flag']=='Overrated'])}")
print(f"Underrated: {len(flagged_products[flagged_products['flag']=='Underrated'])}")
print(f"\nSample:")
print(flagged_products.head(10).to_string(index=False))

Total flagged products: 292

Overrated: 146
Underrated: 146

Sample:
                                                  product_name        brand_name     secondary_category  price_usd price_tier  actual_rating  predicted_rating    gap      flag  review_count  ingredient_score
                                     Tan Build Up Remover Mitt        St. Tropez           Self Tanners      9.000     Budget          4.949             4.212 -0.737 Overrated        59.000             0.000
CLEAR Daily Skin Clearing Treatment with 2.5% Benzoyl Peroxide    Paula'S Choice             Treatments     22.000     Budget          4.833             4.353 -0.480 Overrated        66.000             0.000
                                  Honey Halo Moisturizer Jumbo           Farmacy           Moisturizers     74.000        Mid          4.730             4.251 -0.479 Overrated       100.000             0.093
                               KateCeuticals Lifting Eye Cream   Kate Somerville               Eye 

### Query 4 — Price vs Quality Analysis
How does price tier affect ratings and ingredient quality?

In [18]:
query4 = """
SELECT
    price_tier,
    COUNT(product_id) AS total_products,
    ROUND(AVG(avg_rating), 3) AS avg_rating,
    ROUND(AVG(ingredient_score), 3) AS avg_ingredient_score,
    ROUND(AVG(rating_per_dollar), 4) AS avg_rating_per_dollar,
    ROUND(AVG(five_star_share), 3) AS avg_five_star_share,
    ROUND(AVG(review_count), 0) AS avg_review_count,
    ROUND(MIN(price_usd), 2) AS min_price,
    ROUND(MAX(price_usd), 2) AS max_price
FROM features_ml
WHERE price_tier IS NOT NULL
GROUP BY price_tier
ORDER BY avg_rating DESC
"""

price_quality = pd.read_sql(query4, conn)
print("=== PRICE TIER VS QUALITY ===")
print(price_quality.to_string(index=False))

=== PRICE TIER VS QUALITY ===
price_tier  total_products  avg_rating  avg_ingredient_score  avg_rating_per_dollar  avg_five_star_share  avg_review_count  min_price  max_price
   Premium             284       4.312                 0.100                  0.044                0.661           460.000     76.000    150.000
       Mid            1167       4.280                 0.099                  0.098                0.642           542.000     26.000     75.000
    Luxury              80       4.192                 0.072                  0.018                0.618           207.000    152.000    425.000
    Budget             384       4.122                 0.095                  0.323                0.596           524.000      3.000     25.000


### Query 5 — Hidden Gems
What are the best skincare products I can buy for under $25?

Find products that are:
- Under 25 dollars
- Rated 4.3 or higher
- Have at least 50 reviews so it's not just 3 people

Then sorts by rating_per_dollar 

In [19]:
query5 = """
SELECT
    r.product_name,
    r.brand_name,
    r.secondary_category,
    r.price_usd,
    ROUND(r.avg_rating, 3) AS avg_rating,
    ROUND(r.ingredient_score, 3) AS ingredient_score,
    r.review_count,
    ROUND(f.rating_per_dollar, 4) AS rating_per_dollar,
    ROUND(r.gap, 3) AS gap
FROM results_reliable r
LEFT JOIN features_ml f ON r.product_id = f.product_id
WHERE r.price_usd <= 25
AND r.avg_rating >= 4.3
AND r.review_count >= 50
ORDER BY f.rating_per_dollar DESC
LIMIT 15
"""

hidden_gems = pd.read_sql(query5, conn)
print("=== HIDDEN GEMS (under $25, rated 4.3+, 50+ reviews) ===")
print(hidden_gems.to_string(index=False))

=== HIDDEN GEMS (under $25, rated 4.3+, 50+ reviews) ===
                                  product_name          brand_name     secondary_category  price_usd  avg_rating  ingredient_score  review_count  rating_per_dollar    gap
                 Cleansing & Exfoliating Wipes  Sephora Collection              Cleansers      3.000       4.338             0.111      3837.000              1.446 -0.021
              100% Plant-Derived Hemi-Squalane        The Ordinary             Treatments      5.000       4.399             0.000       296.000              0.880 -0.275
         Mini Fulvic Acid Brightening Cleanser      The Inkey List              Cleansers      5.990       4.418             0.150       347.000              0.738 -0.237
                   100% L-Ascorbic Acid Powder        The Ordinary             Treatments      6.400       4.332             0.500       202.000              0.677 -0.193
                       100% Niacinamide Powder        The Ordinary             Treatment

### Query 6 — Feature Importance Ranking
What objective signals actually drive skincare product quality predictions?

In [20]:
query6 = """
SELECT
    feature,
    ROUND(importance, 4) AS importance,
    ROUND(importance * 100, 2) AS importance_pct
FROM feature_importance
ORDER BY importance DESC
"""

feature_ranks = pd.read_sql(query6, conn)
print("=== FEATURE IMPORTANCE RANKING ===")
print(feature_ranks.to_string(index=False))

=== FEATURE IMPORTANCE RANKING ===
                   feature  importance  importance_pct
         rating_per_dollar       0.284          28.440
           avg_helpfulness       0.251          25.130
                 price_usd       0.153          15.310
            price_rank_pct       0.121          12.110
        review_volume_rank       0.035           3.450
               loves_count       0.030           2.960
        engagement_quality       0.028           2.770
              review_count       0.026           2.580
secondary_category_encoded       0.016           1.640
                       new       0.016           1.580
               value_score       0.016           1.560
          ingredient_score       0.007           0.700
     good_ingredient_count       0.005           0.540
          ingredient_count       0.005           0.480
            good_bad_ratio       0.003           0.330
        price_tier_encoded       0.002           0.230
      bad_ingredient_count    

### Query 7 — Review Volume vs Rating Quality
Do products with more reviews have more reliable ratings?

In [21]:
query7 = """
SELECT
    CASE 
        WHEN review_count < 50 THEN 'Low (under 50)'
        WHEN review_count < 200 THEN 'Medium (50-200)'
        WHEN review_count < 500 THEN 'High (200-500)'
        ELSE 'Very High (500+)'
    END AS review_volume_bucket,
    COUNT(product_id) AS total_products,
    ROUND(AVG(avg_rating), 3) AS avg_rating,
    ROUND(AVG(five_star_share), 3) AS avg_five_star_share,
    ROUND(AVG(one_star_share), 3) AS avg_one_star_share,
    ROUND(AVG(recommendation_rate), 3) AS avg_recommendation_rate
FROM features_ml
GROUP BY review_volume_bucket
ORDER BY AVG(review_count) ASC
"""

review_volume = pd.read_sql(query7, conn)
print("=== REVIEW VOLUME VS RATING QUALITY ===")
print(review_volume.to_string(index=False))



=== REVIEW VOLUME VS RATING QUALITY ===
review_volume_bucket  total_products  avg_rating  avg_five_star_share  avg_one_star_share  avg_recommendation_rate
      Low (under 50)             459       4.085                0.615               0.099                    0.767
     Medium (50-200)             480       4.280                0.648               0.065                    0.849
      High (200-500)             449       4.349                0.651               0.048                    0.874
    Very High (500+)             527       4.280                0.625               0.057                    0.863


In [22]:
conn.close()
print("Database connection closed.")

Database connection closed.
